# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirabdulbaqi/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip -q install duckdb huggingface_hub

In [9]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [10]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully!")

Connected successfully!


In [11]:
con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


## 1. My rule and its reason codes

My baseline rule prioritizes content that may benefit from review based on observed search performance. I will use two main signals: average position and click-through rate (CTR). Content with a relatively poor search position and low CTR receives a higher priority score because it may have opportunities for optimization. This rule provides decision support rather than a final decision and serves as a baseline that a machine learning model should improve upon in later weeks.

**Reason codes:**

* **LOW_CTR** – The content has a relatively low click-through rate.
* **POOR_POSITION** – The content has a relatively poor average search position.
* **REVIEW_CONTENT** – The content should be reviewed for possible optimization.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue (writes the CSV)

The baseline rule calculates a simple priority score using observed CTR and average position. Content with lower CTR and poorer search position receives a higher priority score. Each item is assigned one reason code and one action label. The ranked queue is written to **work/outputs/baseline_action_score.csv** so it can be reviewed and regenerated whenever the notebook is run.


In [13]:
import os
import pandas as pd

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Load a slice of March 2026 data
df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 5000
""").df()

# Calculate CTR
df["ctr"] = (
    df["gsc_clicks"] / df["gsc_impressions"]
).fillna(0)

# Baseline score
df["baseline_score"] = (
    (df["gsc_avg_position"] / 10) +
    ((1 - df["ctr"]) * 5)
)

# Reason code
df["reason_code"] = "LOW_CTR_POOR_POSITION"

# Action label
df["action"] = "REVIEW_CONTENT"

# Rank
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

# Save CSV
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} rows")
print(output_path)

df.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved 5000 rows
work/outputs/baseline_action_score.csv


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ctr,baseline_score,reason_code,action
0,client_73cda7b4e4f265ea,content_3c866dcc3736b629,0,1,136.0,0.0,18.6,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
1,client_73cda7b4e4f265ea,content_e92e8238354f55d1,0,1,99.0,0.0,14.9,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
2,client_73cda7b4e4f265ea,content_17624ba0605baddc,0,1,99.0,0.0,14.9,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
3,client_73cda7b4e4f265ea,content_0b4f8bb8e5510f9b,0,1,98.0,0.0,14.8,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
4,client_73cda7b4e4f265ea,content_97f74022392f589a,0,2,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
5,client_73cda7b4e4f265ea,content_7bb962168fd653c8,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
6,client_73cda7b4e4f265ea,content_09b709c51cfc6ae7,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
7,client_73cda7b4e4f265ea,content_97cd84cf176c63fb,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
8,client_73cda7b4e4f265ea,content_47948d0972e1cb41,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
9,client_73cda7b4e4f265ea,content_a122ce5966fc99fd,0,1,96.0,0.0,14.6,LOW_CTR_POOR_POSITION,REVIEW_CONTENT


## 3. Top-20 review

The highest-ranked content items were selected because they combined low click-through rates with poor average search positions. This baseline rule recommends reviewing these pages before considering content updates or optimization.

For each of the top 10 items:

1. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** The page may have very few impressions, making CTR unstable.
2. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** The page may target a low-demand query where improvements have limited impact.
3. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** Seasonal search patterns may temporarily reduce performance.
4. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** The page may already be scheduled for updates.
5. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** External events may have influenced search behavior.
6. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** Search position alone may not explain user behavior.
7. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** The page may have insufficient historical data.
8. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** Competitor changes may have affected rankings.
9. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** The page may serve a niche audience with naturally low traffic.
10. **Action:** Review Content | **Reason:** LOW_CTR_POOR_POSITION | **Confidence:** Medium | **What could make it wrong?** Additional business context not available in the dataset may change the recommendation.


In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df.head(20)

,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ctr,baseline_score,reason_code,action
0,client_73cda7b4e4f265ea,content_3c866dcc3736b629,0,1,136.000000,0.0,18.600000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
1,client_73cda7b4e4f265ea,content_e92e8238354f55d1,0,1,99.000000,0.0,14.900000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
2,client_73cda7b4e4f265ea,content_17624ba0605baddc,0,1,99.000000,0.0,14.900000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
3,client_73cda7b4e4f265ea,content_0b4f8bb8e5510f9b,0,1,98.000000,0.0,14.800000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
4,client_73cda7b4e4f265ea,content_97f74022392f589a,0,2,97.000000,0.0,14.700000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
5,client_73cda7b4e4f265ea,content_7bb962168fd653c8,0,1,97.000000,0.0,14.700000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
6,client_73cda7b4e4f265ea,content_09b709c51cfc6ae7,0,1,97.000000,0.0,14.700000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
7,client_73cda7b4e4f265ea,content_97cd84cf176c63fb,0,1,97.000000,0.0,14.700000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
8,client_73cda7b4e4f265ea,content_47948d0972e1cb41,0,1,97.000000,0.0,14.700000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
9,client_73cda7b4e4f265ea,content_a122ce5966fc99fd,0,1,96.000000,0.0,14.600000,LOW_CTR_POOR_POSITION,REVIEW_CONTENT


## 4. Weak picks + leakage check

Some recommendations may be weak because they are based on pages with very low impressions, making CTR less reliable. Other pages may perform poorly due to seasonal demand or external factors that are not captured in the data. This baseline rule does not use future information, product decision flags, or any label-derived fields. The score is calculated only from information available at the decision time, reducing the risk of data leakage.


In [15]:
# Check for very low-impression pages in the ranked queue

weak_picks = df[df["gsc_impressions"] <= 5]

print("Weak picks (5 or fewer impressions):", len(weak_picks))

weak_picks.head(10)

Weak picks (5 or fewer impressions): 1636


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ctr,baseline_score,reason_code,action
0,client_73cda7b4e4f265ea,content_3c866dcc3736b629,0,1,136.0,0.0,18.6,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
1,client_73cda7b4e4f265ea,content_e92e8238354f55d1,0,1,99.0,0.0,14.9,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
2,client_73cda7b4e4f265ea,content_17624ba0605baddc,0,1,99.0,0.0,14.9,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
3,client_73cda7b4e4f265ea,content_0b4f8bb8e5510f9b,0,1,98.0,0.0,14.8,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
4,client_73cda7b4e4f265ea,content_97f74022392f589a,0,2,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
5,client_73cda7b4e4f265ea,content_7bb962168fd653c8,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
6,client_73cda7b4e4f265ea,content_09b709c51cfc6ae7,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
7,client_73cda7b4e4f265ea,content_97cd84cf176c63fb,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
8,client_73cda7b4e4f265ea,content_47948d0972e1cb41,0,1,97.0,0.0,14.7,LOW_CTR_POOR_POSITION,REVIEW_CONTENT
9,client_73cda7b4e4f265ea,content_a122ce5966fc99fd,0,1,96.0,0.0,14.6,LOW_CTR_POOR_POSITION,REVIEW_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.